# Agentic Multi-Hop RAG — Colab Experiment
**Before running:** `Runtime → Change runtime type → A100 GPU`

In [1]:
# Cell 1: Mount Drive and set project path
from google.colab import drive
drive.mount('/content/drive')

import os

PROJECT_PATH = '/content/drive/MyDrive/CS572/Final'
os.chdir(PROJECT_PATH)

print('Working directory:', os.getcwd())
print('Files:', os.listdir('.'))


Mounted at /content/drive
Working directory: /content/drive/MyDrive/CS572/Final
Files: ['Final', 'tests', 'src', 'docs', '.git', '.pytest_cache', 'results', '.claude', 'IR Final Project.pdf', 'report.md', 'colab_experiment.ipynb', '.env', '.env.example', 'requirements.txt', 'README.md', '.gitignore', 'results.ipynb', 'final_project.md', 'result.ipynb']


In [2]:
# Cell 2: Install dependencies
!pip install -q -U transformers bitsandbytes accelerate
!pip install -q -U rank-bm25 datasets sentence-transformers python-dotenv google-genai
print('Done.')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 146.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.4/52.4 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 37.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.3/571.3 kB 56.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 783.6/783.6 kB 62.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.6/240.6 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 51.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.47.0, but you have google-auth 2.49.2 which is incompatible.
Done.


In [3]:
# Cell 3: Load .env and log in to Hugging Face
import os
from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv(os.path.join(PROJECT_PATH, '.env'))

hf_token = os.getenv('HF_TOKEN')
gemini_api_key = os.getenv('GEMINI_API_KEY')

if not hf_token:
    raise ValueError('Missing HF_TOKEN in .env')
# Gemini is optional. Only needed when RUN_FRONTIER = True below.

login(token=hf_token)

os.environ['HF_TOKEN'] = hf_token
os.environ['HUGGING_FACE_HUB_TOKEN'] = hf_token
if gemini_api_key:
    os.environ['GEMINI_API_KEY'] = gemini_api_key

print('HF_TOKEN loaded:', bool(hf_token))
print('GEMINI_API_KEY loaded:', bool(gemini_api_key))


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


HF_TOKEN loaded: True
GEMINI_API_KEY loaded: True


In [4]:
# Cell 4: Verify GPU
import torch

print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB' if torch.cuda.is_available() else '')


CUDA available: True
GPU: NVIDIA A100-SXM4-80GB
VRAM: 85.1 GB


In [5]:
# Cell 5: Add project root to Python path and clear stale caches
import sys
import shutil

if PROJECT_PATH not in sys.path:
    sys.path.insert(0, PROJECT_PATH)

for root, dirs, _ in os.walk(PROJECT_PATH):
    for d in dirs:
        if d == '__pycache__':
            shutil.rmtree(os.path.join(root, d))

for mod in list(sys.modules.keys()):
    if mod.startswith('src'):
        del sys.modules[mod]

print('Path set and cache cleared.')


Path set and cache cleared.


In [6]:
# Cell 6: Pilot run. Set RUN_FRONTIER=True only if Gemini has available credits.
from src.run_experiment import run

RUN_FRONTIER = True

pilot_results = run(
    num_samples=5,
    baseline_provider="hf",
    baseline_model_name="meta-llama/Meta-Llama-3.1-8B-Instruct",
    agent_provider="hf",
    agent_model_name="meta-llama/Meta-Llama-3.1-8B-Instruct",
    frontier_provider="gemini" if RUN_FRONTIER else None,
    frontier_model_name="gemini-2.5-flash",
    frontier_num_samples=5,
)

print('Pilot results:', pilot_results)


README.md: 0.00B [00:00, ?B/s]

distractor/train-00000-of-00002.parquet:   0%|          | 0.00/166M [00:00<?, ?B/s]

distractor/train-00001-of-00002.parquet:   0%|          | 0.00/166M [00:00<?, ?B/s]

distractor/validation-00000-of-00001.par(…):   0%|          | 0.00/27.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/90447 [00:00<?, ? examples/s]

...

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample', 'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_t

Running Agentic RAG on 5 samples...


[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

Running frontier Agentic RAG on 5 samples...

=== RESULTS ===
Metric            Baseline    Agentic   Frontier
------------------------------------------------
EM                  0.0000     0.0000     0.6000
F1                  0.4167     0.5167     0.6000
MRR                 0.2667     0.3000     0.3000
NDCG@10             0.3985     0.4617     0.4749
avg_hops            1.0000     3.8000     3.2000
Pilot results: {'metadata': {'num_samples': 5, 'baseline_provider': 'hf', 'baseline_model_name': 'meta-llama/Meta-Llama-3.1-8B-Instruct', 'agent_provider': 'hf', 'agent_model_name': 'meta-llama/Meta-Llama-3.1-8B-Instruct', 'frontier_provider': 'gemini', 'frontier_model_name': 'gemini-2.5-flash', 'frontier_num_samples': 5, 'frontier_completed_samples': 5}, 'baseline': {'EM': 0.0, 'F1': 0.41666666666666663, 'MRR': 0.26666666666666666, 'NDCG@10': 0.3984565739436487, 'avg_hops': 1.0}, 'agentic': {'EM': 0.0, 'F1': 0.5166666666666666, 'MRR': 0.3, 'NDCG@10': 0.4616940730779732, 'avg_hops': 3.8, 

In [7]:
# Cell 7: Full run. Keep RUN_FRONTIER=False unless Gemini has available credits.
from src.run_experiment import run

RUN_FRONTIER = True

results = run(
    num_samples=50,
    baseline_provider="hf",
    baseline_model_name="meta-llama/Meta-Llama-3.1-8B-Instruct",
    agent_provider="hf",
    agent_model_name="meta-llama/Meta-Llama-3.1-8B-Instruct",
    frontier_provider="gemini" if RUN_FRONTIER else None,
    frontier_model_name="gemini-2.5-flash",
    frontier_num_samples=50,  # adjust higher if budget/time allows
)

print('Final results:', results)


[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Running baseline RAG on 50 samples...


[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

Running Agentic RAG on 50 samples...


[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

Running frontier Agentic RAG on 50 samples...

=== RESULTS ===
Metric            Baseline    Agentic   Frontier
------------------------------------------------
EM                  0.3200     0.2200     0.3600
F1                  0.4211     0.3880     0.5105
MRR                 0.3137     0.3538     0.3597
NDCG@10             0.3880     0.4657     0.4829
avg_hops            1.0000     3.4600     2.9400
Final results: {'metadata': {'num_samples': 50, 'baseline_provider': 'hf', 'baseline_model_name': 'meta-llama/Meta-Llama-3.1-8B-Instruct', 'agent_provider': 'hf', 'agent_model_name': 'meta-llama/Meta-Llama-3.1-8B-Instruct', 'frontier_provider': 'gemini', 'frontier_model_name': 'gemini-2.5-flash', 'frontier_num_samples': 50, 'frontier_completed_samples': 50}, 'baseline': {'EM': 0.32, 'F1': 0.42114285714285715, 'MRR': 0.31366666666666665, 'NDCG@10': 0.38799738340253226, 'avg_hops': 1.0}, 'agentic': {'EM': 0.22, 'F1': 0.38799999999999996, 'MRR': 0.3538407703407703, 'NDCG@10': 0.465664284069

In [8]:
# Cell 8: Inspect saved aggregate results
import json

results_path = os.path.join(PROJECT_PATH, 'results', 'experiment_results.json')
with open(results_path) as f:
    saved = json.load(f)

print(json.dumps(saved, indent=2))
print('Saved to:', results_path)


{
  "metadata": {
    "num_samples": 50,
    "baseline_provider": "hf",
    "baseline_model_name": "meta-llama/Meta-Llama-3.1-8B-Instruct",
    "agent_provider": "hf",
    "agent_model_name": "meta-llama/Meta-Llama-3.1-8B-Instruct",
    "frontier_provider": "gemini",
    "frontier_model_name": "gemini-2.5-flash",
    "frontier_num_samples": 50,
    "frontier_completed_samples": 50
  },
  "baseline": {
    "EM": 0.32,
    "F1": 0.42114285714285715,
    "MRR": 0.31366666666666665,
    "NDCG@10": 0.38799738340253226,
    "avg_hops": 1.0
  },
  "agentic": {
    "EM": 0.22,
    "F1": 0.38799999999999996,
    "MRR": 0.3538407703407703,
    "NDCG@10": 0.4656642840697983,
    "avg_hops": 3.46,
    "per_hop_MRR": 0.4550277777777778,
    "per_hop_NDCG@10": 0.5219341103191211
  },
  "frontier_agentic": {
    "EM": 0.36,
    "F1": 0.5105079365079365,
    "MRR": 0.3596991341991342,
    "NDCG@10": 0.48287668236114023,
    "avg_hops": 2.94,
    "per_hop_MRR": 0.4937222222222222,
    "per_hop_NDCG@10"

In [9]:
# Cell 9: White-box breakdowns and case studies
from src.reporting import (
    load_detailed_results,
    print_support_doc_comparison,
    print_support_doc_breakdown,
    print_retrieval_hop_need_comparison,
    print_case_studies,
    print_intermediate_trace,
)

details = load_detailed_results()

# --- Multi-hop vs single-hop need comparison ---
# support_docs_needed: did the question need >=2 gold docs?
print_support_doc_comparison(details)
# retrieval_hop_need: was a single retrieval pass sufficient to find all gold docs?
print_retrieval_hop_need_comparison(details)

# --- Per-system breakdowns ---
print_support_doc_breakdown(details, system_name="baseline")
print_support_doc_breakdown(details, system_name="agentic")
if "frontier_agentic" in details:
    print_support_doc_breakdown(details, system_name="frontier_agentic")

# --- Case studies: agentic ---
print_case_studies(details, system_name="agentic", group_name="successful_multi_doc")
print_case_studies(details, system_name="agentic", group_name="failed_multi_doc")

# --- Case studies: frontier (same question slice for apples-to-apples comparison) ---
if "frontier_agentic" in details:
    print_case_studies(details, system_name="frontier_agentic", group_name="successful_multi_doc")
    print_case_studies(details, system_name="frontier_agentic", group_name="failed_multi_doc")

# --- Intermediate agent traces (hop-by-hop decision log) ---
print_intermediate_trace(details, system_name="agentic", trace_index=0)
print_intermediate_trace(details, system_name="agentic", trace_index=1)
if "frontier_agentic" in details:
    print_intermediate_trace(details, system_name="frontier_agentic", trace_index=0)



=== SUPPORT-DOC NEED COMPARISON ===
Group                    System               Count       EM       F1      MRR    NDCG@10     Hops
--------------------------------------------------------------------------------------------
multiple_support_docs    baseline                50   0.3200   0.4211   0.3137     0.3880     1.00
multiple_support_docs    agentic                 50   0.2200   0.3880   0.3538     0.4657     3.46
multiple_support_docs    frontier_agentic        50   0.3600   0.5105   0.3597     0.4829     2.94

=== RETRIEVAL HOP-NEED COMPARISON ===
Group                    System               Count       EM       F1      MRR    NDCG@10     Hops
--------------------------------------------------------------------------------------------
single_hop_sufficient    baseline                 7   0.7143   0.7143   0.7619     0.8099     1.00
single_hop_sufficient    agentic                  7   0.4286   0.5714   0.7619     0.7872     3.71
single_hop_sufficient    frontier_agentic    


=== SUPPORT-DOC NEED COMPARISON ===
Group                    System               Count       EM       F1      MRR    NDCG@10     Hops
--------------------------------------------------------------------------------------------
multiple_support_docs    baseline                50   0.3200   0.4211   0.3137     0.3880     1.00
multiple_support_docs    agentic                 50   0.2200   0.3880   0.3538     0.4657     3.46
multiple_support_docs    frontier_agentic        50   0.3600   0.5105   0.3597     0.4829     2.94

=== RETRIEVAL HOP-NEED COMPARISON ===
Group                    System               Count       EM       F1      MRR    NDCG@10     Hops
--------------------------------------------------------------------------------------------
single_hop_sufficient    baseline                 7   0.7143   0.7143   0.7619     0.8099     1.00
single_hop_sufficient    agentic                  7   0.4286   0.5714   0.7619     0.7872     3.71
single_hop_sufficient    frontier_agentic         7   0.5714   0.7347   0.7619     0.7745     2.43
needs_followup_hops      baseline                43   0.2558   0.3734   0.2407     0.3193     1.00
needs_followup_hops      agentic                 43   0.1860   0.3581   0.2874     0.4133     3.42
needs_followup_hops      frontier_agentic        43   0.3256   0.4740   0.2942     0.4354     3.02

=== BASELINE SUPPORT-DOC BREAKDOWN ===
Group                      Count       EM       F1      MRR    NDCG@10     Hops
---------------------------------------------------------------------------
multiple_support_docs         50   0.3200   0.4211   0.3137     0.3880     1.00

=== AGENTIC SUPPORT-DOC BREAKDOWN ===
Group                      Count       EM       F1      MRR    NDCG@10     Hops
---------------------------------------------------------------------------
multiple_support_docs         50   0.2200   0.3880   0.3538     0.4657     3.46

=== FRONTIER_AGENTIC SUPPORT-DOC BREAKDOWN ===
Group                      Count       EM       F1      MRR    NDCG@10     Hops
---------------------------------------------------------------------------
multiple_support_docs         50   0.3600   0.5105   0.3597     0.4829     2.94

=== AGENTIC SUCCESSFUL_MULTI_DOC ===

Case 1: 2014 S/S is the debut album of a South Korean boy group that was formed by who?
Gold: YG Entertainment
Pred: YG Entertainment.
Hops: 2
Supporting titles: 2014 S/S, Winner (band)
Sub-queries: ['2014 S/S is the debut album of a South Korean boy group that was formed by who?', 'YG Entertainment']
  Decision hop 1: sufficient=False next='YG Entertainment' final=None
  Decision hop 2: sufficient=True next='YG Entertainment' final=None
  Hop query: 2014 S/S is the debut album of a South Korean boy group that was formed by who?
  Docs: ['List of awards and nominations received by Shinee', '2014 S/S', 'List of songs written by Ravi', 'BTS discography', 'Madtown']
  Hop query: YG Entertainment
  Docs: ['2014 S/S', 'Winner (band)', 'History (band)', 'Seventeen discography', 'SF9 (band)']

Case 2: Who is older, Annie Morton or Terry Richardson?
Gold: Terry Richardson
Pred: Terry Richardson.
Hops: 4
Supporting titles: Annie Morton, Terry Richardson
Sub-queries: ['Who is older, Annie Morton or Terry Richardson?', "Annie Morton's birthdate and Terry Richardson's birthdate", "Annie Morton's birth year and Terry Richardson's birth year", "Annie Morton's birthdate or Terry Richardson's birthdate", "Terry Richardson's birthdate or Annie Morton's birthdate"]
  Decision hop 1: sufficient=False next="Annie Morton's birthdate and Terry Richardson's birthdate" final=None
  Decision hop 2: sufficient=False next="Annie Morton's birth year and Terry Richardson's birth year" final=None
  Decision hop 3: sufficient=False next="Annie Morton's birthdate or Terry Richardson's birthdate" final=None
  Decision hop 4: sufficient=False next="Terry Richardson's birthdate or Annie Morton's birthdate" final=None
  Hop query: Who is older, Annie Morton or Terry Richardson?
  Docs: ['Lady Gaga x Terry Richardson', 'Annie Morton', 'Gumbo (PJ Morton album)', 'Terry Richardson', 'Shadowshaper']
  Hop query: Annie Morton's birthdate and Terry Richardson's birthdate
  Docs: ['Annie Morton', 'Lady Gaga x Terry Richardson', 'Gumbo (PJ Morton album)', 'Madonna (book)', 'Terry Richardson']
  Hop query: Annie Morton's birth year and Terry Richardson's birth year
  Docs: ['Annie Morton', 'The Wonderful Thing About Tiggers', 'Lady Gaga x Terry Richardson', 'Gumbo (PJ Morton album)', 'Madonna (book)']
  Hop query: Annie Morton's birthdate or Terry Richardson's birthdate
  Docs: ['Annie Morton', 'Lady Gaga x Terry Richardson', 'Madonna (book)', 'Terry Richardson', 'Amanda Lepore']

=== AGENTIC FAILED_MULTI_DOC ===

Case 1: Were Scott Derrickson and Ed Wood of the same nationality?
Gold: yes
Pred: Yes, American.
Hops: 3
Supporting titles: Scott Derrickson, Ed Wood
Sub-queries: ['Were Scott Derrickson and Ed Wood of the same nationality?', 'Scott Derrickson nationality', 'Scott Derrickson birthplace']
  Decision hop 1: sufficient=False next='Scott Derrickson nationality' final=None
  Decision hop 2: sufficient=False next='Scott Derrickson birthplace' final=None
  Decision hop 3: sufficient=True next='Scott Derrickson birthplace' final=None
  Hop query: Were Scott Derrickson and Ed Wood of the same nationality?
  Docs: ['Tyler Bates', 'Woodson, Arkansas', 'Doctor Strange (2016 film)', 'Gone in 60 Seconds (2000 film)', 'Ed Wood (film)']
  Hop query: Scott Derrickson nationality
  Docs: ['Scott Derrickson', 'Sinister (film)', 'Doctor Strange (2016 film)', 'Adam Collis', 'Deliver Us from Evil (2014 film)']
  Hop query: Scott Derrickson birthplace
  Docs: ['Scott Derrickson', 'Sinister (film)', 'Doctor Strange (2016 film)', 'Adam Collis', 'Deliver Us from Evil (2014 film)']

Case 2: What government position was held by the woman who portrayed Corliss Archer in the film Kiss and Tell?
Gold: Chief of Protocol
Pred: Village accountant.
Hops: 4
Supporting titles: Kiss and Tell (1945 film), Shirley Temple
Sub-queries: ['What government position was held by the woman who portrayed Corliss Archer in the film Kiss and Tell?', "Shirley Temple's filmography or biography", 'Shirley Temple filmography or roles in the 1940s', 'Corliss Archer film Kiss and Tell government position', "Janet Waldo's roles in radio shows"]
  Decision hop 1: sufficient=False next="Shirley Temple's filmography or biography" final=None
  Decision hop 2: sufficient=False next='Shirley Temple filmography or roles in the 1940s' final=None
  Decision hop 3: sufficient=False next='Corliss Archer film Kiss and Tell government position' final=None
  Decision hop 4: sufficient=False next="Janet Waldo's roles in radio shows" final=None
  Hop query: What government position was held by the woman who portrayed Corliss Archer in the film Kiss and Tell?
  Docs: ['Charles Craft', 'Village accountant', 'A Kiss for Corliss', '2014 Bahrain Grand Prix', 'City of Angels (film)']
  Hop query: Shirley Temple's filmography or biography
  Docs: ['Madonna (book)', 'Shirley Temple', 'Andre Norton Award', 'Lead programmer', 'Front Row (software)']
  Hop query: Shirley Temple filmography or roles in the 1940s
  Docs: ['Bernhard Bötel', 'Kate Higgins', 'Kansas City jazz', 'English Electric Canberra', 'Giuseppe Verdi (film)']
  Hop query: Corliss Archer film Kiss and Tell government position
  Docs: ['A Kiss for Corliss', 'Kiss and Tell (1945 film)', 'Charles Craft', 'Village accountant', 'Janet Waldo']

=== FRONTIER_AGENTIC SUCCESSFUL_MULTI_DOC ===

Case 1: Were Scott Derrickson and Ed Wood of the same nationality?
Gold: yes
Pred: Yes
Hops: 3
Supporting titles: Scott Derrickson, Ed Wood
Sub-queries: ['Were Scott Derrickson and Ed Wood of the same nationality?', 'Scott Derrickson nationality', 'Ed Wood nationality']
  Decision hop 1: sufficient=False next='Scott Derrickson nationality' final=None
  Decision hop 2: sufficient=False next='Ed Wood nationality' final=None
  Decision hop 3: sufficient=True next='Were Scott Derrickson and Ed Wood of the same nationality?' final='Yes'
  Hop query: Were Scott Derrickson and Ed Wood of the same nationality?
  Docs: ['Tyler Bates', 'Woodson, Arkansas', 'Doctor Strange (2016 film)', 'Gone in 60 Seconds (2000 film)', 'Ed Wood (film)']
  Hop query: Scott Derrickson nationality
  Docs: ['Scott Derrickson', 'Sinister (film)', 'Doctor Strange (2016 film)', 'Adam Collis', 'Deliver Us from Evil (2014 film)']
  Hop query: Ed Wood nationality
  Docs: ['Ed Wood (film)', 'Woodson, Arkansas', 'Conrad Brooks', 'Ed Wood', 'Hackers (film)']

Case 2: What science fantasy young adult series, told in first person, has a set of companion books narrating the stories of enslaved worlds and alien species?
Gold: Animorphs
Pred: Animorphs
Hops: 3
Supporting titles: The Hork-Bajir Chronicles, Animorphs
Sub-queries: ['What science fantasy young adult series, told in first person, has a set of companion books narrating the stories of enslaved worlds and alien species?', 'science fantasy young adult series first person enslaved worlds alien species companion books', 'Animorphs companion books enslaved worlds alien species']
  Decision hop 1: sufficient=False next='science fantasy young adult series first person enslaved worlds alien species companion books' final=None
  Decision hop 2: sufficient=False next='Animorphs companion books enslaved worlds alien species' final=None
  Decision hop 3: sufficient=True next='What science fantasy young adult series, told in first person, has a set of companion books narrating the stories of enslaved worlds and alien species?' final='Animorphs'
  Hop query: What science fantasy young adult series, told in first person, has a set of companion books narrating the stories of enslaved worlds and alien species?
  Docs: ['Andre Norton Award', 'List of Square Enix companion books', 'The Hork-Bajir Chronicles', 'Science Fantasy (magazine)', 'Left Behind: The Kids']
  Hop query: science fantasy young adult series first person enslaved worlds alien species companion books
  Docs: ['Andre Norton Award', 'List of Square Enix companion books', 'Science Fantasy (magazine)', 'Left Behind: The Kids', 'Animorphs']
  Hop query: Animorphs companion books enslaved worlds alien species
  Docs: ['List of Square Enix companion books', 'The Hork-Bajir Chronicles', 'Animorphs', 'Victoria Hanley', 'Science Fantasy (magazine)']

=== FRONTIER_AGENTIC FAILED_MULTI_DOC ===

Case 1: What government position was held by the woman who portrayed Corliss Archer in the film Kiss and Tell?
Gold: Chief of Protocol
Pred: Ambassador to Ghana and Czechoslovakia
Hops: 3
Supporting titles: Kiss and Tell (1945 film), Shirley Temple
Sub-queries: ['What government position was held by the woman who portrayed Corliss Archer in the film Kiss and Tell?', 'who portrayed Corliss Archer in Kiss and Tell', 'Shirley Temple government position']
  Decision hop 1: sufficient=False next='who portrayed Corliss Archer in Kiss and Tell' final=None
  Decision hop 2: sufficient=False next='Shirley Temple government position' final=None
  Decision hop 3: sufficient=True next='What government position was held by the woman who portrayed Corliss Archer in the film Kiss and Tell?' final='Ambassador to Ghana and Czechoslovakia'
  Hop query: What government position was held by the woman who portrayed Corliss Archer in the film Kiss and Tell?
  Docs: ['Charles Craft', 'Village accountant', 'A Kiss for Corliss', '2014 Bahrain Grand Prix', 'City of Angels (film)']
  Hop query: who portrayed Corliss Archer in Kiss and Tell
  Docs: ['A Kiss for Corliss', 'Charles Craft', 'Florence Nagle', 'Arnold Schwarzenegger filmography', 'Kiss and Tell (1945 film)']
  Hop query: Shirley Temple government position
  Docs: ['Village accountant', 'Shirley Temple', 'Lord High Treasurer', 'Secretary of State for Constitutional Affairs', 'Robert R']

Case 2: Are the Laleli Mosque and Esma Sultan Mansion located in the same neighborhood?
Gold: no
Pred: Information not available
Hops: 4
Supporting titles: Laleli Mosque, Esma Sultan Mansion
Sub-queries: ['Are the Laleli Mosque and Esma Sultan Mansion located in the same neighborhood?', 'Laleli Mosque location', 'Esma Sultan Mansion location', 'Laleli Mosque location', 'Esma Sultan Mansion location']
  Decision hop 1: sufficient=False next='Laleli Mosque location' final=None
  Decision hop 2: sufficient=False next='Esma Sultan Mansion location' final=None
  Decision hop 3: sufficient=False next='Laleli Mosque location' final=None
  Decision hop 4: sufficient=False next='Esma Sultan Mansion location' final=None
  Hop query: Are the Laleli Mosque and Esma Sultan Mansion located in the same neighborhood?
  Docs: ['University of Missouri–Kansas City', 'Djamaâ el Kebir', 'Sultan Ahmed Mosque', 'Esma Sultan Mansion', 'Esma Sultan (daughter of Abdul Hamid I)']
  Hop query: Laleli Mosque location
  Docs: ['Sultan Ahmed Mosque', 'Djamaâ el Kebir', 'Laleli Mosque', 'New York Society of Model Engineers', 'Androscoggin Bank Colisée']
  Hop query: Esma Sultan Mansion location
  Docs: ['Esma Sultan Mansion', 'Esma Sultan (daughter of Abdul Hamid I)', 'Esma Sultan (daughter of Ahmed III)', 'Esma Sultan (daughter of Abdülaziz)', 'Sultan Ahmed Mosque']
  Hop query: Laleli Mosque location
  Docs: ['Sultan Ahmed Mosque', 'Djamaâ el Kebir', 'Laleli Mosque', 'New York Society of Model Engineers', 'Androscoggin Bank Colisée']

=== AGENTIC TRACE 0 ===
Question: Were Scott Derrickson and Ed Wood of the same nationality?
Gold: yes
Pred: Yes, American.
Requires multiple docs: True
Num hops: 3
Sub-queries: ['Were Scott Derrickson and Ed Wood of the same nationality?', 'Scott Derrickson nationality', 'Scott Derrickson birthplace']

Hop 1 query: Were Scott Derrickson and Ed Wood of the same nationality?
Top docs: ['Tyler Bates', 'Woodson, Arkansas', 'Doctor Strange (2016 film)', 'Gone in 60 Seconds (2000 film)', 'Ed Wood (film)']
Decision: {'sub_query': 'Scott Derrickson nationality', 'sufficient': False}
Sufficient: False

Hop 2 query: Scott Derrickson nationality
Top docs: ['Scott Derrickson', 'Sinister (film)', 'Doctor Strange (2016 film)', 'Adam Collis', 'Deliver Us from Evil (2014 film)']
Decision: {'sub_query': 'Scott Derrickson birthplace', 'sufficient': False}
Sufficient: False

Hop 3 query: Scott Derrickson birthplace
Top docs: ['Scott Derrickson', 'Sinister (film)', 'Doctor Strange (2016 film)', 'Adam Collis', 'Deliver Us from Evil (2014 film)']
Decision: {'sub_query': 'Scott Derrickson birthplace', 'sufficient': False}
Sufficient: True

=== AGENTIC TRACE 1 ===
Question: What government position was held by the woman who portrayed Corliss Archer in the film Kiss and Tell?
Gold: Chief of Protocol
Pred: Village accountant.
Requires multiple docs: True
Num hops: 4
Sub-queries: ['What government position was held by the woman who portrayed Corliss Archer in the film Kiss and Tell?', "Shirley Temple's filmography or biography", 'Shirley Temple filmography or roles in the 1940s', 'Corliss Archer film Kiss and Tell government position', "Janet Waldo's roles in radio shows"]

Hop 1 query: What government position was held by the woman who portrayed Corliss Archer in the film Kiss and Tell?
Top docs: ['Charles Craft', 'Village accountant', 'A Kiss for Corliss', '2014 Bahrain Grand Prix', 'City of Angels (film)']
Decision: {'sub_query': "Shirley Temple's filmography or biography", 'sufficient': False}
Sufficient: False

Hop 2 query: Shirley Temple's filmography or biography
Top docs: ['Madonna (book)', 'Shirley Temple', 'Andre Norton Award', 'Lead programmer', 'Front Row (software)']
Decision: {'sub_query': 'Shirley Temple filmography or roles in the 1940s', 'sufficient': False}
Sufficient: False

Hop 3 query: Shirley Temple filmography or roles in the 1940s
Top docs: ['Bernhard Bötel', 'Kate Higgins', 'Kansas City jazz', 'English Electric Canberra', 'Giuseppe Verdi (film)']
Decision: {'sub_query': 'Corliss Archer film Kiss and Tell government position', 'sufficient': False}
Sufficient: False

Hop 4 query: Corliss Archer film Kiss and Tell government position
Top docs: ['A Kiss for Corliss', 'Kiss and Tell (1945 film)', 'Charles Craft', 'Village accountant', 'Janet Waldo']
Decision: {'sub_query': "Janet Waldo's roles in radio shows", 'sufficient': False}
Sufficient: False

=== FRONTIER_AGENTIC TRACE 0 ===
Question: Were Scott Derrickson and Ed Wood of the same nationality?
Gold: yes
Pred: Yes
Requires multiple docs: True
Num hops: 3
Sub-queries: ['Were Scott Derrickson and Ed Wood of the same nationality?', 'Scott Derrickson nationality', 'Ed Wood nationality']

Hop 1 query: Were Scott Derrickson and Ed Wood of the same nationality?
Top docs: ['Tyler Bates', 'Woodson, Arkansas', 'Doctor Strange (2016 film)', 'Gone in 60 Seconds (2000 film)', 'Ed Wood (film)']
Decision: {'sub_query': 'Scott Derrickson nationality', 'sufficient': False}
Sufficient: False

Hop 2 query: Scott Derrickson nationality
Top docs: ['Scott Derrickson', 'Sinister (film)', 'Doctor Strange (2016 film)', 'Adam Collis', 'Deliver Us from Evil (2014 film)']
Decision: {'sub_query': 'Ed Wood nationality', 'sufficient': False}
Sufficient: False

Hop 3 query: Ed Wood nationality
Top docs: ['Ed Wood (film)', 'Woodson, Arkansas', 'Conrad Brooks', 'Ed Wood', 'Hackers (film)']
Decision: {'sub_query': '', 'sufficient': True, 'final_answer': 'Yes'}
Sufficient: True

In [10]:
# Cell 10: Frontier model ceiling
# Gemini-2.5-Pro is the performance ceiling — if the open-source agentic model
# matches or beats it, the bottleneck is the retrieval loop, not model capacity.
from src.reporting import load_detailed_results, print_frontier_ceiling_summary

details = load_detailed_results()
print_frontier_ceiling_summary(details)



=== FRONTIER MODEL CEILING ===
System                                       EM       F1      MRR    NDCG@10   Avg Hops
--------------------------------------------------------------------------------------
Baseline RAG (LLaMA-3.1-8B)              0.3200   0.4211   0.3137     0.3880       1.00
Agentic RAG (LLaMA-3.1-8B)               0.2200   0.3880   0.3538     0.4657       3.46
Frontier Agentic (Gemini-2.5-Flash)      0.3600   0.5105   0.3597     0.4829       2.94

  Frontier vs Agentic  EM      : +0.1400
  Frontier vs Agentic  F1      : +0.1225
  Frontier vs Agentic  MRR     : +0.0059
  Frontier vs Agentic  NDCG@10 : +0.0172

  Observation: Frontier improves EM by +0.1400 over agentic.


In [11]:
# Cell 11: White-box traces — full retrieved text at each hop
# Shows WHY the agent succeeded or failed: what text was actually retrieved.
from src.reporting import load_detailed_results, print_full_trace_with_text

details = load_detailed_results()

agentic_traces = details["agentic"]["traces"]
multi_success = [t for t in agentic_traces if t["requires_multiple_documents"] and t["exact_match"]]
multi_fail    = [t for t in agentic_traces if t["requires_multiple_documents"] and not t["exact_match"]]

print("\n" + "="*70)
print("WHITE-BOX: Successful multi-hop (Agentic)")
print("="*70)
if multi_success:
    print_full_trace_with_text(multi_success[0])

print("\n" + "="*70)
print("WHITE-BOX: Failed multi-hop (Agentic)")
print("="*70)
if multi_fail:
    print_full_trace_with_text(multi_fail[0])

if "frontier_agentic" in details:
    frontier_traces = details["frontier_agentic"]["traces"]
    f_success = [t for t in frontier_traces if t["requires_multiple_documents"] and t["exact_match"]]
    f_fail    = [t for t in frontier_traces if t["requires_multiple_documents"] and not t["exact_match"]]

    print("\n" + "="*70)
    print("WHITE-BOX: Successful multi-hop (Frontier — Gemini-2.5-Pro)")
    print("="*70)
    if f_success:
        print_full_trace_with_text(f_success[0])

    print("\n" + "="*70)
    print("WHITE-BOX: Failed multi-hop (Frontier — Gemini-2.5-Pro)")
    print("="*70)
    if f_fail:
        print_full_trace_with_text(f_fail[0])



WHITE-BOX: Successful multi-hop (Agentic)

Question    : 2014 S/S is the debut album of a South Korean boy group that was formed by who?
Gold        : YG Entertainment
Pred        : YG Entertainment.
Correct     : yes
Hops        : 2
Multi-doc   : True
Sub-queries : ['2014 S/S is the debut album of a South Korean boy group that was formed by who?', 'YG Entertainment']

  --- Hop 1: '2014 S/S is the debut album of a South Korean boy group that was formed by who?' ---
  Decision  : sufficient=False, next_query='YG Entertainment'
  [List of awards and nominations received by Shinee] List of awards and nominations received by Shinee. South Korean boy group Shinee have received several awards and nominations for their music work.  The group was formed by S.M. Entertainment in 2008 and released their f...
  [2014 S/S] 2014 S/S. 2014 S/S is the debut album of South Korean group WINNER.  It was released on August 12, 2014 by the group's record label, YG Entertainment.  The members were credit


======================================================================
WHITE-BOX: Successful multi-hop (Agentic)
======================================================================

Question    : 2014 S/S is the debut album of a South Korean boy group that was formed by who?
Gold        : YG Entertainment
Pred        : YG Entertainment.
Correct     : yes
Hops        : 2
Multi-doc   : True
Sub-queries : ['2014 S/S is the debut album of a South Korean boy group that was formed by who?', 'YG Entertainment']

  --- Hop 1: '2014 S/S is the debut album of a South Korean boy group that was formed by who?' ---
  Decision  : sufficient=False, next_query='YG Entertainment'
  [List of awards and nominations received by Shinee] List of awards and nominations received by Shinee. South Korean boy group Shinee have received several awards and nominations for their music work.  The group was formed by S.M. Entertainment in 2008 and released their f...
  [2014 S/S] 2014 S/S. 2014 S/S is the debut album of South Korean group WINNER.  It was released on August 12, 2014 by the group's record label, YG Entertainment.  The members were credited for writing the lyrics and composing the m...
  [List of songs written by Ravi] List of songs written by Ravi. Ravi is a South Korean rapper, songwriter and producer, signed under Jellyfish Entertainment.  He began his career as a rapper in 2012 in the South Korean boy group VIXX, and later formed V...
  [BTS discography] BTS discography. The following is the discography of South Korean boy group BTS.  The group debuted in South Korea on June 2013 with single album, "2 Cool 4 Skool", at number 5 on South Korean Week 31 Gaon Weekly Chart. ...
  [Madtown] Madtown. Madtown (Hangul: 매드타운 ), often stylized as MADTOWN, is a South Korean boy group formed in 2014 by J. Tune Camp.  The group consists of Moos, Daewon, Lee Geon, Jota, Heo Jun, Buffy and H.O.  Their debut album, "M...

  --- Hop 2: 'YG Entertainment' ---
  Decision  : sufficient=True, next_query='YG Entertainment'
  [2014 S/S] 2014 S/S. 2014 S/S is the debut album of South Korean group WINNER.  It was released on August 12, 2014 by the group's record label, YG Entertainment.  The members were credited for writing the lyrics and composing the m...
  [Winner (band)] Winner (band). Winner (Hangul: 위너), often stylized as WINNER, is a South Korean boy group formed in 2013 by YG Entertainment and debuted in 2014.  It currently consists of four members, Jinwoo, Seunghoon, Mino and Seungy...
  [History (band)] History (band). History (Korean: 히스토리 ) was a South Korean boy group formed by LOEN Entertainment in 2013.  They debuted on April 26, 2013 with "Dreamer", featuring the narration of their labelmate IU.  They were LOEN En...
  [Seventeen discography] Seventeen discography. This is the discography of South Korean boy group Seventeen.  Seventeen (Hangul: 세븐틴), also stylized as SEVENTEEN or SVT, is a South Korean boy group formed by Pledis Entertainment in 2015.  They h...
  [SF9 (band)] SF9 (band). SF9 (Korean: 에스에프나인 ; shortened from Sensational Feeling 9) is a South Korean boy group formed by FNC Entertainment.  SF9 is the company's first dance boy group to ever debut.  SF9 debuted on October 5, 2016 ...

======================================================================
WHITE-BOX: Failed multi-hop (Agentic)
======================================================================

Question    : Were Scott Derrickson and Ed Wood of the same nationality?
Gold        : yes
Pred        : Yes, American.
Correct     : no
Hops        : 3
Multi-doc   : True
Sub-queries : ['Were Scott Derrickson and Ed Wood of the same nationality?', 'Scott Derrickson nationality', 'Scott Derrickson birthplace']

  --- Hop 1: 'Were Scott Derrickson and Ed Wood of the same nationality?' ---
  Decision  : sufficient=False, next_query='Scott Derrickson nationality'
  [Tyler Bates] Tyler Bates. Tyler Bates (born June 5, 1965) is an American musician, music producer, and composer for films, television, and video games.  Much of his work is in the action and horror film genres, with films like "Dawn ...
  [Woodson, Arkansas] Woodson, Arkansas. Woodson is a census-designated place (CDP) in Pulaski County, Arkansas, in the United States.  Its population was 403 at the 2010 census.  It is part of the Little Rock–North Little Rock–Conway Metropo...
  [Doctor Strange (2016 film)] Doctor Strange (2016 film). Doctor Strange is a 2016 American superhero film based on the Marvel Comics character of the same name, produced by Marvel Studios and distributed by Walt Disney Studios Motion Pictures.  It i...
  [Gone in 60 Seconds (2000 film)] Gone in 60 Seconds (2000 film). Gone in 60 Seconds is a 2000 American action heist film, starring Nicolas Cage, Angelina Jolie, Giovanni Ribisi, Christopher Eccleston, Robert Duvall, Vinnie Jones, and Will Patton.  The f...
  [Ed Wood (film)] Ed Wood (film). Ed Wood is a 1994 American biographical period comedy-drama film directed and produced by Tim Burton, and starring Johnny Depp as cult filmmaker Ed Wood.  The film concerns the period in Wood's life when ...

  --- Hop 2: 'Scott Derrickson nationality' ---
  Decision  : sufficient=False, next_query='Scott Derrickson birthplace'
  [Scott Derrickson] Scott Derrickson. Scott Derrickson (born July 16, 1966) is an American director, screenwriter and producer.  He lives in Los Angeles, California.  He is best known for directing horror films such as "Sinister", "The Exor...
  [Sinister (film)] Sinister (film). Sinister is a 2012 supernatural horror film directed by Scott Derrickson and written by Derrickson and C. Robert Cargill.  It stars Ethan Hawke as fictional true-crime writer Ellison Oswalt who discovers...
  [Doctor Strange (2016 film)] Doctor Strange (2016 film). Doctor Strange is a 2016 American superhero film based on the Marvel Comics character of the same name, produced by Marvel Studios and distributed by Walt Disney Studios Motion Pictures.  It i...
  [Adam Collis] Adam Collis. Adam Collis is an American filmmaker and actor.  He attended the Duke University from 1986 to 1990 and the University of California, Los Angeles from 2007 to 2010.  He also studied cinema at the University o...
  [Deliver Us from Evil (2014 film)] Deliver Us from Evil (2014 film). Deliver Us from Evil is a 2014 American supernatural horror film directed by Scott Derrickson and produced by Jerry Bruckheimer.  The film is officially based on a 2001 non-fiction book ...

  --- Hop 3: 'Scott Derrickson birthplace' ---
  Decision  : sufficient=True, next_query='Scott Derrickson birthplace'
  [Scott Derrickson] Scott Derrickson. Scott Derrickson (born July 16, 1966) is an American director, screenwriter and producer.  He lives in Los Angeles, California.  He is best known for directing horror films such as "Sinister", "The Exor...
  [Sinister (film)] Sinister (film). Sinister is a 2012 supernatural horror film directed by Scott Derrickson and written by Derrickson and C. Robert Cargill.  It stars Ethan Hawke as fictional true-crime writer Ellison Oswalt who discovers...
  [Doctor Strange (2016 film)] Doctor Strange (2016 film). Doctor Strange is a 2016 American superhero film based on the Marvel Comics character of the same name, produced by Marvel Studios and distributed by Walt Disney Studios Motion Pictures.  It i...
  [Adam Collis] Adam Collis. Adam Collis is an American filmmaker and actor.  He attended the Duke University from 1986 to 1990 and the University of California, Los Angeles from 2007 to 2010.  He also studied cinema at the University o...
  [Deliver Us from Evil (2014 film)] Deliver Us from Evil (2014 film). Deliver Us from Evil is a 2014 American supernatural horror film directed by Scott Derrickson and produced by Jerry Bruckheimer.  The film is officially based on a 2001 non-fiction book ...

======================================================================
WHITE-BOX: Successful multi-hop (Frontier — Gemini-2.5-Pro)
======================================================================

Question    : Were Scott Derrickson and Ed Wood of the same nationality?
Gold        : yes
Pred        : Yes
Correct     : yes
Hops        : 3
Multi-doc   : True
Sub-queries : ['Were Scott Derrickson and Ed Wood of the same nationality?', 'Scott Derrickson nationality', 'Ed Wood nationality']

  --- Hop 1: 'Were Scott Derrickson and Ed Wood of the same nationality?' ---
  Decision  : sufficient=False, next_query='Scott Derrickson nationality'
  [Tyler Bates] Tyler Bates. Tyler Bates (born June 5, 1965) is an American musician, music producer, and composer for films, television, and video games.  Much of his work is in the action and horror film genres, with films like "Dawn ...
  [Woodson, Arkansas] Woodson, Arkansas. Woodson is a census-designated place (CDP) in Pulaski County, Arkansas, in the United States.  Its population was 403 at the 2010 census.  It is part of the Little Rock–North Little Rock–Conway Metropo...
  [Doctor Strange (2016 film)] Doctor Strange (2016 film). Doctor Strange is a 2016 American superhero film based on the Marvel Comics character of the same name, produced by Marvel Studios and distributed by Walt Disney Studios Motion Pictures.  It i...
  [Gone in 60 Seconds (2000 film)] Gone in 60 Seconds (2000 film). Gone in 60 Seconds is a 2000 American action heist film, starring Nicolas Cage, Angelina Jolie, Giovanni Ribisi, Christopher Eccleston, Robert Duvall, Vinnie Jones, and Will Patton.  The f...
  [Ed Wood (film)] Ed Wood (film). Ed Wood is a 1994 American biographical period comedy-drama film directed and produced by Tim Burton, and starring Johnny Depp as cult filmmaker Ed Wood.  The film concerns the period in Wood's life when ...

  --- Hop 2: 'Scott Derrickson nationality' ---
  Decision  : sufficient=False, next_query='Ed Wood nationality'
  [Scott Derrickson] Scott Derrickson. Scott Derrickson (born July 16, 1966) is an American director, screenwriter and producer.  He lives in Los Angeles, California.  He is best known for directing horror films such as "Sinister", "The Exor...
  [Sinister (film)] Sinister (film). Sinister is a 2012 supernatural horror film directed by Scott Derrickson and written by Derrickson and C. Robert Cargill.  It stars Ethan Hawke as fictional true-crime writer Ellison Oswalt who discovers...
  [Doctor Strange (2016 film)] Doctor Strange (2016 film). Doctor Strange is a 2016 American superhero film based on the Marvel Comics character of the same name, produced by Marvel Studios and distributed by Walt Disney Studios Motion Pictures.  It i...
  [Adam Collis] Adam Collis. Adam Collis is an American filmmaker and actor.  He attended the Duke University from 1986 to 1990 and the University of California, Los Angeles from 2007 to 2010.  He also studied cinema at the University o...
  [Deliver Us from Evil (2014 film)] Deliver Us from Evil (2014 film). Deliver Us from Evil is a 2014 American supernatural horror film directed by Scott Derrickson and produced by Jerry Bruckheimer.  The film is officially based on a 2001 non-fiction book ...

  --- Hop 3: 'Ed Wood nationality' ---
  Decision  : sufficient=True, next_query='Were Scott Derrickson and Ed Wood of the same nationality?'
  [Ed Wood (film)] Ed Wood (film). Ed Wood is a 1994 American biographical period comedy-drama film directed and produced by Tim Burton, and starring Johnny Depp as cult filmmaker Ed Wood.  The film concerns the period in Wood's life when ...
  [Woodson, Arkansas] Woodson, Arkansas. Woodson is a census-designated place (CDP) in Pulaski County, Arkansas, in the United States.  Its population was 403 at the 2010 census.  It is part of the Little Rock–North Little Rock–Conway Metropo...
  [Conrad Brooks] Conrad Brooks. Conrad Brooks (born Conrad Biedrzycki on January 3, 1931 in Baltimore, Maryland) is an American actor.  He moved to Hollywood, California in 1948 to pursue a career in acting.  He got his start in movies a...
  [Ed Wood] Ed Wood. Edward Davis Wood Jr. (October 10, 1924 – December 10, 1978) was an American filmmaker, actor, writer, producer, and director....
  [Hackers (film)] Hackers (film). Hackers is a 1995 American crime film directed by Iain Softley and starring Jonny Lee Miller, Angelina Jolie, Renoly Santiago, Matthew Lillard, Jesse Bradford, Lorraine Bracco, and Fisher Stevens.  The fi...

======================================================================
WHITE-BOX: Failed multi-hop (Frontier — Gemini-2.5-Pro)
======================================================================

Question    : What government position was held by the woman who portrayed Corliss Archer in the film Kiss and Tell?
Gold        : Chief of Protocol
Pred        : Ambassador to Ghana and Czechoslovakia
Correct     : no
Hops        : 3
Multi-doc   : True
Sub-queries : ['What government position was held by the woman who portrayed Corliss Archer in the film Kiss and Tell?', 'who portrayed Corliss Archer in Kiss and Tell', 'Shirley Temple government position']

  --- Hop 1: 'What government position was held by the woman who portrayed Corliss Archer in the film Kiss and Tell?' ---
  Decision  : sufficient=False, next_query='who portrayed Corliss Archer in Kiss and Tell'
  [Charles Craft] Charles Craft. Charles Craft (May 9, 1902 – September 19, 1968) was an English-born American film and television editor.  Born in the county of Hampshire in England on May 9, 1902, Craft would enter the film industry in ...
  [Village accountant] Village accountant. The Village Accountant (variously known as "Patwari", "Talati", "Patel", "Karnam", "Adhikari", "Shanbogaru","Patnaik" etc.) is an administrative government position found in rural parts of the Indian ...
  [A Kiss for Corliss] A Kiss for Corliss. A Kiss for Corliss is a 1949 American comedy film directed by Richard Wallace and written by Howard Dimsdale.  It stars Shirley Temple in her final starring role as well as her final film appearance. ...
  [2014 Bahrain Grand Prix] 2014 Bahrain Grand Prix. The 2014 Bahrain Grand Prix (formally the 2014 Formula 1 Gulf Air Bahrain Grand Prix) was a Formula One motor race held on 6 April at the Bahrain International Circuit in Sakhir, Bahrain.  It was...
  [City of Angels (film)] City of Angels (film). City of Angels is a 1998 American romantic fantasy film directed by Brad Silberling and starring Nicolas Cage and Meg Ryan.  Set in Los Angeles, California, the film is a loose remake of Wim Wender...

  --- Hop 2: 'who portrayed Corliss Archer in Kiss and Tell' ---
  Decision  : sufficient=False, next_query='Shirley Temple government position'
  [A Kiss for Corliss] A Kiss for Corliss. A Kiss for Corliss is a 1949 American comedy film directed by Richard Wallace and written by Howard Dimsdale.  It stars Shirley Temple in her final starring role as well as her final film appearance. ...
  [Charles Craft] Charles Craft. Charles Craft (May 9, 1902 – September 19, 1968) was an English-born American film and television editor.  Born in the county of Hampshire in England on May 9, 1902, Craft would enter the film industry in ...
  [Florence Nagle] Florence Nagle. Florence Nagle (26 October 1894 – 30 October 1988) was a trainer and breeder of racehorses, a breeder of pedigree dogs, and an active feminist.  Nagle purchased her first Irish Wolfhound in 1913, and went...
  [Arnold Schwarzenegger filmography] Arnold Schwarzenegger filmography. Arnold Schwarzenegger is an actor who has appeared in over 30 films, and has also ventured into directing and producing.  He began his acting career primarily with small roles in film a...
  [Kiss and Tell (1945 film)] Kiss and Tell (1945 film). Kiss and Tell is a 1945 American comedy film starring then 17-year-old Shirley Temple as Corliss Archer.  In the film, two teenage girls cause their respective parents much concern when they st...

  --- Hop 3: 'Shirley Temple government position' ---
  Decision  : sufficient=True, next_query='What government position was held by the woman who portrayed Corliss Archer in the film Kiss and Tell?'
  [Village accountant] Village accountant. The Village Accountant (variously known as "Patwari", "Talati", "Patel", "Karnam", "Adhikari", "Shanbogaru","Patnaik" etc.) is an administrative government position found in rural parts of the Indian ...
  [Shirley Temple] Shirley Temple. Shirley Temple Black (April 23, 1928 – February 10, 2014) was an American actress, singer, dancer, businesswoman, and diplomat who was Hollywood's number one box-office draw as a child actress from 1935 t...
  [Lord High Treasurer] Lord High Treasurer. The post of Lord High Treasurer or Lord Treasurer was an English government position and has been a British government position since the Acts of Union of 1707.  A holder of the post would be the thi...
  [Secretary of State for Constitutional Affairs] Secretary of State for Constitutional Affairs. The office of Secretary of State for Constitutional Affairs was a British Government position, created in 2003.  Certain functions of the Lord Chancellor which related to th...
  [Robert R] Robert R. Hood. Robert R. Hood is an American government official who currently serves as Assistant Secretary of Defense for Legislative Affairs.  Hood was previously vice president for government affairs for CH2M.  Past...

In [12]:
# Cell 12: Raw trace inspection (single sample) — manual deep dive
details = load_detailed_results()
trace = details["agentic"]["traces"][0]

print("Question:", trace["question"])
print("Gold:", trace["gold_answer"])
print("Pred:", trace["predicted_answer"])
print("Sub-queries:", trace["sub_queries"])
print("Requires multiple docs:", trace["requires_multiple_documents"])
print("Num hops:", trace["num_hops"])

for hop in trace["per_hop"]:
    print("\nHop query:", hop["query"])
    for doc in hop["docs"]:
        print("-", doc["title"], "|", doc["text_preview"])


Question: Were Scott Derrickson and Ed Wood of the same nationality?
Gold: yes
Pred: Yes, American.
Sub-queries: ['Were Scott Derrickson and Ed Wood of the same nationality?', 'Scott Derrickson nationality', 'Scott Derrickson birthplace']
Requires multiple docs: True
Num hops: 3

Hop query: Were Scott Derrickson and Ed Wood of the same nationality?
- Tyler Bates | Tyler Bates. Tyler Bates (born June 5, 1965) is an American musician, music producer, and composer for films, television, and video games.  Much of his work is in the action and horror film genres, with films like "Dawn 
- Woodson, Arkansas | Woodson, Arkansas. Woodson is a census-designated place (CDP) in Pulaski County, Arkansas, in the United States.  Its population was 403 at the 2010 census.  It is part of the Little Rock–North Little Rock–Conway Metropo
- Doctor Strange (2016 film) | Doctor Strange (2016 film). Doctor Strange is a 2016 American superhero film based on the Marvel Comics character of the same name, produc